# Honey Yield Predictive Modeling Pipeline

This notebook develops a reproducible DuckDB workflow for preparing hive sensor data for predictive modeling. The project objective is to predict next-day hive weight change using historical hive measurements, environmental conditions, and available colony-event information.

The workflow follows a layered structure:

```text
Raw source data
      │
      ▼
DuckDB anchor tables
      │
      ▼
Clean measurement tables
      │
      ▼
Feature engineering
      │
      ▼
Modeling dataset
      │
      ▼
Predictive models

> Notebook Role: This is the clean project implementation notebook. Exploratory tests, debugging cells, and experimental modeling decisions are developed separately before being incorporated here.
>
> Development Note: This notebook was developed and validated using a representative subset of the daily hive measurements to enable rapid iteration during pipeline development. The workflow is designed to scale to the complete archive with minimal modification once feature engineering and modeling logic have been verified.

## 1. Project Setup

This section imports the required Python libraries and defines reusable project paths.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
working_dir = Path.cwd()
project_root = working_dir.parent

data_dir = project_root / "Data"
documentation_dir = project_root / "Documentation"
figures_dir = project_root / "Figures"
exports_dir = project_root / "Exports"

print(f"Working directory: {working_dir}")
print(f"Project root: {project_root}")

Working directory: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Stephanie_Work
Project root: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model


In [3]:
database_path = data_dir / "honey.duckdb"

con = duckdb.connect(str(database_path))

print(f"Connected to DuckDB: {database_path}")

Connected to DuckDB: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\honey.duckdb


In [4]:
temp_dir = project_root / "Data" / "duckdb_temp"
temp_dir.mkdir(parents=True, exist_ok=True)

con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}';")
con.execute("SET preserve_insertion_order = false;")

print("DuckDB resource settings configured.")

DuckDB resource settings configured.


In [5]:
required_directories = [
    data_dir,
    documentation_dir,
    figures_dir,
    exports_dir
]

for directory in required_directories:
    print(f"{directory.name}: {directory.exists()}")

Data: True
Documentation: False
Figures: False
Exports: False


## 2. DuckDB Connection

DuckDB provides the analytical database layer for the project. Persistent tables stored in the database remain available after the notebook kernel is restarted.

In [6]:
database_path = data_dir / "honey_mini.duckdb"

con = duckdb.connect(str(database_path))

print(f"Connected to DuckDB: {database_path}")

Connected to DuckDB: C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\honey_mini.duckdb


In [7]:
con.execute("""
SHOW TABLES;
""").df()

,name


## 3. Source Data Verification

This section verifies that the expected source data are available before any project tables are created. The project uses daily hive measurements as the initial modeling grain.

In [8]:
daily_data_dir = data_dir / "Daily_Only"

print(daily_data_dir)

C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\Daily_Only


In [9]:
import re

all_csv_files = list(daily_data_dir.rglob("*.csv"))

daily_csv_files = [
    file
    for file in all_csv_files
    if re.search(
        r"/\d{4}_d/[^/]+\.csv$",
        file.as_posix()
    )
]

print(f"All CSV files: {len(all_csv_files):,}")
print(f"Daily files: {len(daily_csv_files):,}")

All CSV files: 453
Daily files: 151


## 4. Raw Anchor Table

The raw anchor table preserves the source measurements with minimal modification. It is created only when it does not already exist. Downstream cleaning and feature engineering should never overwrite the raw anchor.

In [10]:
daily_csv_pattern = str(daily_data_dir / "**" / "*.csv")

print(daily_csv_pattern)

C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\Daily_Only\**\*.csv


In [11]:
escaped_daily_files = [
    file.as_posix().replace("'", "''")
    for file in daily_csv_files
]

daily_file_list_sql = ",\n".join(
    f"'{file}'"
    for file in escaped_daily_files
)

existing_tables = set(
    con.execute("SHOW TABLES;").df()["name"]
)

if "honey_daily_raw" in existing_tables:
    print("Using existing raw anchor table: honey_daily_raw")
else:
    print("Creating raw anchor table: honey_daily_raw")

    con.execute(f"""
    CREATE TABLE honey_daily_raw AS
    SELECT *
    FROM read_csv_auto(
        [
            {daily_file_list_sql}
        ],
        filename = TRUE,
        union_by_name = TRUE,
        all_varchar = TRUE
    );
    """)

    print("Created raw anchor table: honey_daily_raw")

Creating raw anchor table: honey_daily_raw
Created raw anchor table: honey_daily_raw


In [12]:
con.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT filename) AS source_file_count
FROM honey_daily_raw;
""").df()

,row_count,source_file_count
0,29172,151


In [13]:
con.execute("""
DESCRIBE honey_daily_raw;
""").df()

,column_name,column_type,null,key,default,extra
0,column00,VARCHAR,YES,None,None,None
1,X.1,VARCHAR,YES,None,None,None
2,time,VARCHAR,YES,None,None,None
3,X,VARCHAR,YES,None,None,None
4,t_i_1,VARCHAR,YES,None,None,None
5,t_i_2,VARCHAR,YES,None,None,None
6,t_i_3,VARCHAR,YES,None,None,None
7,t_i_4,VARCHAR,YES,None,None,None
8,t_i_5,VARCHAR,YES,None,None,None
9,t_o,VARCHAR,YES,None,None,None


## 5. Clean Measurement Table

This section converts raw text values into analytical data types and replaces source placeholders such as `"NA"` with proper SQL `NULL` values.

In [14]:
con.execute("""
CREATE OR REPLACE TABLE honey_daily_clean AS
SELECT
    TRY_CAST(time AS TIMESTAMP) AS measurement_time,

    TRY_CAST(NULLIF(weight_kg, 'NA') AS DOUBLE) AS weight_kg,
    TRY_CAST(NULLIF(weight_delta, 'NA') AS DOUBLE) AS source_weight_delta_kg,

    TRY_CAST(NULLIF(t_i_1, 'NA') AS DOUBLE) AS internal_temp_1,
    TRY_CAST(NULLIF(t_i_2, 'NA') AS DOUBLE) AS internal_temp_2,
    TRY_CAST(NULLIF(t_i_3, 'NA') AS DOUBLE) AS internal_temp_3,
    TRY_CAST(NULLIF(t_i_4, 'NA') AS DOUBLE) AS internal_temp_4,
    TRY_CAST(NULLIF(t_o, 'NA') AS DOUBLE) AS outside_temp,

    TRY_CAST(NULLIF(h, 'NA') AS DOUBLE) AS humidity,
    TRY_CAST(NULLIF(p, 'NA') AS DOUBLE) AS pressure,

    TRY_CAST(NULLIF(lat, 'NA') AS DOUBLE) AS latitude,
    TRY_CAST(NULLIF(lon, 'NA') AS DOUBLE) AS longitude,

    filename AS source_filename

FROM honey_daily_raw;
""");

print("Created clean table: honey_daily_clean")

Created clean table: honey_daily_clean


In [15]:
con.execute("""
DESCRIBE honey_daily_clean;
""").df()

,column_name,column_type,null,key,default,extra
0,measurement_time,TIMESTAMP,YES,None,None,None
1,weight_kg,DOUBLE,YES,None,None,None
2,source_weight_delta_kg,DOUBLE,YES,None,None,None
3,internal_temp_1,DOUBLE,YES,None,None,None
4,internal_temp_2,DOUBLE,YES,None,None,None
5,internal_temp_3,DOUBLE,YES,None,None,None
6,internal_temp_4,DOUBLE,YES,None,None,None
7,outside_temp,DOUBLE,YES,None,None,None
8,humidity,DOUBLE,YES,None,None,None
9,pressure,DOUBLE,YES,None,None,None


In [16]:
con.execute("""
DESCRIBE honey_daily_clean;
""").df()

,column_name,column_type,null,key,default,extra
0,measurement_time,TIMESTAMP,YES,None,None,None
1,weight_kg,DOUBLE,YES,None,None,None
2,source_weight_delta_kg,DOUBLE,YES,None,None,None
3,internal_temp_1,DOUBLE,YES,None,None,None
4,internal_temp_2,DOUBLE,YES,None,None,None
5,internal_temp_3,DOUBLE,YES,None,None,None
6,internal_temp_4,DOUBLE,YES,None,None,None
7,outside_temp,DOUBLE,YES,None,None,None
8,humidity,DOUBLE,YES,None,None,None
9,pressure,DOUBLE,YES,None,None,None


In [17]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM honey_daily_raw) AS raw_rows,
    (SELECT COUNT(*) FROM honey_daily_clean) AS clean_rows;
""").df()

,raw_rows,clean_rows
0,29172,29172


In [18]:
con.execute("""
WITH daily_counts AS (
    SELECT
        source_filename,
        CAST(measurement_time AS DATE) AS measurement_date,
        COUNT(*) AS readings_per_day
    FROM honey_daily_clean
    GROUP BY
        source_filename,
        CAST(measurement_time AS DATE)
)

SELECT
    MIN(readings_per_day) AS minimum_readings_per_day,
    MAX(readings_per_day) AS maximum_readings_per_day,
    COUNT(*) AS hive_date_records
FROM daily_counts;
""").df()

,minimum_readings_per_day,maximum_readings_per_day,hive_date_records
0,1,1,29172


## 6. Daily Hive Summary

This section creates one standardized observation per hive per calendar day. It also derives a consistent hive identifier from the source filename.

In [19]:
con.execute("""
CREATE OR REPLACE TABLE honey_daily_summary AS
SELECT
    TRY_CAST(
        regexp_extract(
            source_filename,
            '([^/\\\\]+)\\.csv$',
            1
        ) AS INTEGER
    ) AS hive_id,

    CAST(measurement_time AS DATE) AS measurement_date,

    weight_kg AS end_of_day_weight_kg,

    internal_temp_1 AS avg_internal_temp_1,
    internal_temp_2 AS avg_internal_temp_2,
    internal_temp_3 AS avg_internal_temp_3,
    internal_temp_4 AS avg_internal_temp_4,
    outside_temp AS avg_outside_temp,
    humidity AS avg_humidity,
    pressure AS avg_pressure,

    latitude,
    longitude,
    source_filename

FROM honey_daily_clean;
""");

print("Created daily summary table: honey_daily_summary")

Created daily summary table: honey_daily_summary


In [20]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM honey_daily_raw) AS raw_rows,
    (SELECT COUNT(*) FROM honey_daily_clean) AS clean_rows,
    (SELECT COUNT(*) FROM honey_daily_summary) AS summary_rows,
    (SELECT COUNT(DISTINCT source_filename)
     FROM honey_daily_clean) AS source_files;
""").df()

,raw_rows,clean_rows,summary_rows,source_files
0,29172,29172,29172,151


In [21]:
print(database_path)

C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\honey_mini.duckdb


In [22]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM honey_daily_raw) AS raw_rows,
    (SELECT COUNT(*) FROM honey_daily_clean) AS clean_rows,
    (SELECT COUNT(*) FROM honey_daily_summary) AS summary_rows,
    (SELECT COUNT(DISTINCT source_filename)
     FROM honey_daily_clean) AS source_files;
""").df()

,raw_rows,clean_rows,summary_rows,source_files
0,29172,29172,29172,151


In [23]:
con.execute("""
SELECT
    current_setting('threads') AS threads,
    current_setting('memory_limit') AS memory_limit,
    current_setting('temp_directory') AS temp_directory,
    current_setting('preserve_insertion_order') AS preserve_insertion_order;
""").df()

,threads,memory_limit,temp_directory,preserve_insertion_order
0,8,9.3 GiB,C:\Users\sfnor\Miniconda3\envs\pandas_book\hon...,True


In [24]:
print(daily_data_dir)

C:\Users\sfnor\Miniconda3\envs\pandas_book\honey-yield-predictive-model\Data\Daily_Only


In [25]:
len(list(daily_data_dir.rglob("*.csv")))

453

## 7. Feature Engineering

Historical features are calculated independently within each hive using SQL window functions. These variables represent recent hive conditions and temporal trends that may help predict the next day's weight change.

In [26]:
con.execute("""
CREATE OR REPLACE TABLE honey_feature_candidates AS

WITH weight_windows AS (
    SELECT
        hive_id,
        measurement_date,
        end_of_day_weight_kg,

        LAG(measurement_date) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
        ) AS previous_observation_date,

        LEAD(measurement_date) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
        ) AS next_observation_date,

        LAG(end_of_day_weight_kg) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
        ) AS previous_day_weight_kg,

        LEAD(end_of_day_weight_kg) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
        ) AS next_day_weight_kg,

        AVG(end_of_day_weight_kg) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3_day_weight_kg,

        AVG(end_of_day_weight_kg) OVER (
            PARTITION BY hive_id
            ORDER BY measurement_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7_day_weight_kg

    FROM honey_daily_summary
),

engineered_weights AS (
    SELECT
        *,

        DATE_DIFF(
            'day',
            previous_observation_date,
            measurement_date
        ) AS days_since_previous_observation,

        DATE_DIFF(
            'day',
            measurement_date,
            next_observation_date
        ) AS days_to_next_observation,

        end_of_day_weight_kg
            - previous_day_weight_kg
            AS previous_day_weight_change_kg,

        next_day_weight_kg
            - end_of_day_weight_kg
            AS target_next_day_weight_change_kg

    FROM weight_windows
)

SELECT
    e.hive_id,
    e.measurement_date,
    e.end_of_day_weight_kg,
    e.previous_observation_date,
    e.next_observation_date,
    e.previous_day_weight_kg,
    e.next_day_weight_kg,
    e.rolling_3_day_weight_kg,
    e.rolling_7_day_weight_kg,
    e.days_since_previous_observation,
    e.days_to_next_observation,
    e.previous_day_weight_change_kg,
    e.target_next_day_weight_change_kg,

    s.avg_internal_temp_1,
    s.avg_internal_temp_2,
    s.avg_internal_temp_3,
    s.avg_internal_temp_4,
    s.avg_outside_temp,
    s.avg_humidity,
    s.avg_pressure,
    s.latitude,
    s.longitude,
    s.source_filename,

    EXTRACT(YEAR FROM e.measurement_date) AS year,
    EXTRACT(MONTH FROM e.measurement_date) AS month,
    EXTRACT(DOY FROM e.measurement_date) AS day_of_year

FROM engineered_weights AS e

LEFT JOIN honey_daily_summary AS s
    ON e.hive_id = s.hive_id
   AND e.measurement_date = s.measurement_date;
""")

print("Created feature candidate table: honey_feature_candidates")

Created feature candidate table: honey_feature_candidates


In [27]:
con.execute("""
SELECT
    COUNT(*) AS feature_rows,
    COUNT(DISTINCT hive_id) AS hive_count,
    MIN(measurement_date) AS first_date,
    MAX(measurement_date) AS last_date
FROM honey_feature_candidates;
""").df()

,feature_rows,hive_count,first_date,last_date
0,29172,78,2019-06-24,2022-12-31


> **Window-Function Logic**
>
> `PARTITION BY hive_id` restarts each calculation for every hive. `LAG()` retrieves the previous available observation, while `LEAD()` retrieves the following observation. Calendar-gap fields are retained so that only true consecutive-day records are used in the final modeling dataset.

## 8. Modeling Dataset

The final modeling table retains records with valid previous-day and next-day observations. Extreme weight changes are flagged for analysis rather than automatically removed.

In [28]:
con.execute("""
CREATE OR REPLACE TABLE honey_model AS
SELECT
    *,

    CASE
        WHEN ABS(target_next_day_weight_change_kg) > 5
        THEN TRUE
        ELSE FALSE
    END AS extreme_weight_change_flag

FROM honey_feature_candidates

WHERE days_since_previous_observation = 1
  AND days_to_next_observation = 1
  AND end_of_day_weight_kg IS NOT NULL
  AND previous_day_weight_kg IS NOT NULL
  AND next_day_weight_kg IS NOT NULL
  AND end_of_day_weight_kg > 0
  AND previous_day_weight_kg > 0
  AND next_day_weight_kg > 0
  AND target_next_day_weight_change_kg IS NOT NULL;
""");

print("Created final modeling table: honey_model")

Created final modeling table: honey_model


In [29]:
con.execute("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT hive_id) AS hives,
    MIN(measurement_date) AS first_date,
    MAX(measurement_date) AS last_date,
    AVG(target_next_day_weight_change_kg) AS average_target_change
FROM honey_model;
""").df()

,observations,hives,first_date,last_date,average_target_change
0,26215,78,2019-06-25,2022-12-30,0.002042


In [30]:
con.execute("""
SELECT
    COUNT(*) AS duplicate_hive_dates
FROM (
    SELECT
        hive_id,
        measurement_date,
        COUNT(*) AS row_count
    FROM honey_model
    GROUP BY
        hive_id,
        measurement_date
    HAVING COUNT(*) > 1
);
""").df()

,duplicate_hive_dates
0,0


## 9. Preliminary Analysis

This section evaluates whether the available data can support the project's prediction objective and investigates the distribution of next-day hive weight changes.

In [31]:
con.execute("""
SELECT
    extreme_weight_change_flag,
    COUNT(*) AS observations,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM honey_model
GROUP BY extreme_weight_change_flag
ORDER BY extreme_weight_change_flag;
""").df()

,extreme_weight_change_flag,observations,percentage
0,False,25770,98.3
1,True,445,1.7


In [32]:
con.execute("""
SELECT
    AVG(target_next_day_weight_change_kg) AS mean_change,
    MEDIAN(target_next_day_weight_change_kg) AS median_change,
    STDDEV(target_next_day_weight_change_kg) AS standard_deviation,
    MIN(target_next_day_weight_change_kg) AS minimum_change,
    MAX(target_next_day_weight_change_kg) AS maximum_change
FROM honey_model;
""").df()

,mean_change,median_change,standard_deviation,minimum_change,maximum_change
0,0.002042,-0.044655,1.840522,-65.32294,39.098176
